# HTA-MAC Phase 2B three-seed confirmation

This notebook runs only the frozen confirmation experiment:

- shared Branching Dueling C51;
- slot budget 12;
- optimizer seeds 2299, 3299, and 4299;
- each seed starts from its own registered budget-12 checkpoint;
- 125 episodes by default, with 25 frozen development clusters;
- feature scaling, hybrid trajectory-order loss, and concavity loss frozen before results;
- automatic FP32 versus BF16 benchmark;
- Drive backup after every seed;
- paired mid-episode hybrid counterfactual audit and predeclared acceptance checks.

This is development confirmation, not Phase 4 and not a publication result.


In [ ]:
# User settings
BUNDLE_PATH = "/content/HTA_MAC_Phase2B_Confirmation_Bundle_20260803.zip"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/HTA_MAC_Phase2B_Confirmation"
SEEDS = [2299, 3299, 4299]
EPISODES = 125
BF16_MINIMUM_SPEEDUP = 1.10
DOWNLOAD_RESULTS_WHEN_COMPLETE = True

assert 100 <= EPISODES <= 150, "Keep the frozen confirmation between 100 and 150 episodes."
assert SEEDS == [2299, 3299, 4299], "Do not change the registered confirmation seeds."


In [ ]:
# Hardware check, robust ZIP discovery, safe extraction, and manifest verification.
import glob, hashlib, json, os, platform, shutil, stat, sys, zipfile
from pathlib import Path, PurePosixPath
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

expected_sha256 = "85b62a91654b91c525592c6e5fb87927223f8eb57ac53f836e327ab92804e6eb"
stem = "HTA_MAC_Phase2B_Confirmation_Bundle_20260803"
patterns = [f"/content/{stem}.zip", f"/content/{stem}*.zip"]

def matches(patterns):
    found = []
    for pattern in patterns:
        found.extend(Path(item) for item in glob.glob(pattern))
    return sorted({item.resolve() for item in found if item.is_file()})

candidates = matches(patterns)
if not candidates:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_patterns = [
        f"/content/drive/MyDrive/{stem}.zip",
        f"/content/drive/MyDrive/{stem}*.zip",
    ]
    candidates = matches(drive_patterns)
if not candidates:
    explicit = Path(BUNDLE_PATH)
    if explicit.is_file():
        candidates = [explicit.resolve()]
if not candidates:
    raise FileNotFoundError("Upload the supplied Phase 2B confirmation ZIP to /content or MyDrive.")

bundle = max(candidates, key=lambda item: item.stat().st_mtime_ns)
actual_sha256 = hashlib.sha256(bundle.read_bytes()).hexdigest()
if actual_sha256 != expected_sha256:
    raise RuntimeError(f"Bundle SHA-256 mismatch: {actual_sha256} != {expected_sha256}")
print("Using bundle:", bundle)
print("SHA256:", actual_sha256)

workspace = Path("/content/stage2")
if workspace.exists():
    if workspace.resolve() != Path("/content/stage2"):
        raise RuntimeError("Unsafe workspace cleanup target")
    shutil.rmtree(workspace)

with zipfile.ZipFile(bundle, "r") as archive:
    bad = archive.testzip()
    if bad:
        raise RuntimeError(f"Corrupt ZIP entry: {bad}")
    names = [entry.filename.replace("\\", "/") for entry in archive.infolist()]
    if "stage2/COLAB_PHASE2B_MANIFEST.json" not in set(names):
        raise RuntimeError("This is not the expected Phase 2B bundle")
    if len(names) != len(set(names)):
        raise RuntimeError("Archive contains colliding normalized paths")
    for entry, name in zip(archive.infolist(), names):
        normalized = PurePosixPath(name)
        parts = normalized.parts
        if normalized.is_absolute() or not parts or parts[0] != "stage2":
            raise RuntimeError(f"Unexpected archive entry: {entry.filename}")
        relative = parts[1:]
        if not relative:
            continue
        if any(part in ("", ".", "..") for part in relative):
            raise RuntimeError(f"Unsafe archive entry: {entry.filename}")
        if ((entry.external_attr >> 16) & 0o170000) == stat.S_IFLNK:
            raise RuntimeError(f"Symlink archive entry rejected: {entry.filename}")
        target = workspace.joinpath(*relative)
        if entry.is_dir() or name.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)

repo = workspace / "hta-mac"
upstream = workspace / "final_repo"
manifest_path = workspace / "COLAB_PHASE2B_MANIFEST.json"
assert repo.is_dir(), repo
assert upstream.is_dir(), upstream
manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
failures = []
for record in manifest["files"]:
    target = workspace.joinpath(*PurePosixPath(record["path"]).parts)
    if not target.is_file():
        failures.append("missing:" + record["path"])
        continue
    if target.stat().st_size != record["bytes"]:
        failures.append("size:" + record["path"])
        continue
    if hashlib.sha256(target.read_bytes()).hexdigest() != record["sha256"]:
        failures.append("sha256:" + record["path"])
if failures:
    raise RuntimeError("Bundle manifest verification failed: " + str(failures[:10]))
print("MANIFEST_FILES_VERIFIED:", len(manifest["files"]))

os.chdir(repo)
sys.path.insert(0, str(repo))


In [ ]:
# Install only declared runtime dependencies.
import subprocess, sys
packages = [
    "gymnasium>=0.29,<2",
    "torch-geometric>=2.4,<3",
    "numpy>=1.24",
    "scipy>=1.10",
    "pytest>=7",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed.")


In [ ]:
# Preflight and real-model FP32/BF16 benchmark.
import time
import numpy as np
import torch
from agents.branching_dqn import BranchingAgentConfig, BranchingDQNAgent

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU before training.")
device = "cuda"
torch.set_float32_matmul_precision("high")

subprocess.run([sys.executable, "-m", "compileall", "-q", str(repo), str(upstream)], check=True)
subprocess.run([
    sys.executable, "-B", "-m", "pytest", "validation",
    "--ignore=validation/test_phase3_policies.py",
    "-q", "-p", "no:cacheprovider"
], cwd=repo, check=True)

sample_checkpoint = repo / "inputs" / "registered_shared_b12_seed2299.pt"
payload = torch.load(sample_checkpoint, map_location="cpu", weights_only=False)

def make_agent(precision):
    cfg_data = dict(payload["config"])
    cfg_data["precision"] = precision
    agent = BranchingDQNAgent(BranchingAgentConfig(**cfg_data), device=device)
    agent.online.load_state_dict(payload["online_state_dict"])
    return agent

states = torch.randn(8, 100, 50, device=device)
masks = torch.ones(8, 100, dtype=torch.bool, device=device)

def benchmark(precision):
    agent = make_agent(precision)
    optimizer = torch.optim.Adam(agent.online.parameters(), lr=1e-5)
    times = []
    finite = True
    for index in range(16):
        torch.cuda.synchronize()
        started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        q = agent.q_values_tensor(states, masks)
        loss = q.float().square().mean()
        finite = finite and bool(torch.isfinite(loss))
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        if index >= 6:
            times.append(time.perf_counter() - started)
    return float(np.median(times)), finite

fp32_seconds, fp32_finite = benchmark("fp32")
bf16_supported = bool(torch.cuda.is_bf16_supported())
bf16_seconds, bf16_finite = (float("inf"), False)
if bf16_supported:
    bf16_seconds, bf16_finite = benchmark("bf16")
speedup = fp32_seconds / bf16_seconds if bf16_seconds < float("inf") else 0.0
PRECISION = (
    "bf16"
    if bf16_supported and bf16_finite and speedup >= BF16_MINIMUM_SPEEDUP
    else "fp32"
)
print({
    "gpu": torch.cuda.get_device_name(0),
    "fp32_seconds": fp32_seconds,
    "bf16_supported": bf16_supported,
    "bf16_seconds": bf16_seconds,
    "bf16_speedup": speedup,
    "selected_precision": PRECISION,
})


In [ ]:
# Restore completed runs, train each missing seed, audit, and back up immediately.
import json, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

drive_root = Path(DRIVE_OUTPUT_DIR)
drive_runs = drive_root / "runs"
drive_runs.mkdir(parents=True, exist_ok=True)
local_phase2 = repo / "outputs" / "phase2"
local_phase2.mkdir(parents=True, exist_ok=True)

def run_name(seed):
    return f"phase2b_confirm_shared_b12_seed{seed}_{EPISODES}ep"

def local_run(seed):
    return local_phase2 / run_name(seed)

def decision_pass(path):
    try:
        return bool(json.loads(path.read_text(encoding="utf-8"))["confirmation_pass"])
    except Exception:
        return False

# Restore Drive state first.
for seed in SEEDS:
    source = drive_runs / run_name(seed)
    if source.is_dir():
        shutil.copytree(source, local_run(seed), dirs_exist_ok=True)

for index, seed in enumerate(SEEDS):
    run_dir = local_run(seed)
    decision = run_dir / "confirmation_decision.json"
    if decision_pass(decision):
        print(f"SKIP_CONFIRMATION_PASS seed={seed} run={run_name(seed)}")
        continue
    initial = repo / "inputs" / f"registered_shared_b12_seed{seed}.pt"
    command = [
        sys.executable, "-B", "experiments/run_phase2b_confirmation.py",
        "--seed", str(seed),
        "--episodes", str(EPISODES),
        "--device", "cuda",
        "--precision", PRECISION,
        "--initial-checkpoint", str(initial),
        "--run-name", run_name(seed),
    ]
    print(f"START {index + 1}/3 seed={seed} precision={PRECISION}", flush=True)
    started = time.time()
    completed = subprocess.run(command, cwd=repo, check=False)
    runtime = {
        "seed": seed,
        "precision": PRECISION,
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "elapsed_seconds": time.time() - started,
        "return_code": completed.returncode,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "command": command,
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "colab_runtime.json").write_text(json.dumps(runtime, indent=2), encoding="utf-8")
    shutil.copytree(run_dir, drive_runs / run_name(seed), dirs_exist_ok=True)
    print(f"BACKUP_COMPLETE seed={seed} minutes={runtime['elapsed_seconds']/60:.1f}", flush=True)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Seed {seed} failed a predeclared confirmation check. "
            "Its complete evidence was backed up; stop and inspect rather than continuing."
        )


In [ ]:
# Aggregate the three decisions and save the final registry to Drive.
import json
from datetime import datetime, timezone

rows = []
for seed in SEEDS:
    decision_path = local_run(seed) / "confirmation_decision.json"
    if not decision_path.is_file():
        rows.append({"seed": seed, "status": "missing"})
    else:
        decision = json.loads(decision_path.read_text(encoding="utf-8"))
        rows.append({
            "seed": seed,
            "status": decision["status"],
            "confirmation_pass": decision["confirmation_pass"],
            "predeclared_checks": decision["predeclared_checks"],
            "run_name": decision["run_name"],
        })

complete = len(rows) == 3 and all(row.get("confirmation_pass") for row in rows)
registry = {
    "schema_version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "Phase 2B shared-branching budget-12 three-seed confirmation",
    "episodes": EPISODES,
    "precision": PRECISION,
    "seeds": SEEDS,
    "runs": rows,
    "complete_and_passed": complete,
    "scope": "Development confirmation only; held-out Phase 3/4 evaluation remains required.",
}
registry_path = drive_root / "PHASE2B_CONFIRMATION_REGISTRY.json"
registry_path.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print(json.dumps(registry, indent=2))
print("PHASE2B_THREE_SEED_CONFIRMATION_PASS=", complete)


In [ ]:
# Create one downloadable ZIP after all three seeds pass.
import shutil
from pathlib import Path

if DOWNLOAD_RESULTS_WHEN_COMPLETE:
    registry = json.loads((drive_root / "PHASE2B_CONFIRMATION_REGISTRY.json").read_text())
    if not registry["complete_and_passed"]:
        raise RuntimeError("Results are incomplete or a seed failed; inspect Drive evidence first.")
    archive_base = Path("/content/HTA_MAC_Phase2B_Confirmation_Results_20260803")
    archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=drive_root))
    print("RESULT_ZIP:", archive)
    print("RESULT_ZIP_BYTES:", archive.stat().st_size)
    from google.colab import files
    files.download(str(archive))
else:
    print("Automatic result download disabled; all evidence remains in", drive_root)
